# SegFormer fine-tune on Colab — 차선 세그 (CVAT polyline zip 업로드)

Phase 1: 차선 세그용 SegFormer fine-tune.
**3 클래스 + 배경** → id2label = {0: background, 1: left-solid, 2: right-solid, 3: center-dashed}

이 id2label 순서는 `data_pipeline/extract_labels.py`의 `SegFormerLaneSeg`가
기대하는 것과 정확히 일치해야 한다 (배경=0, 의미 클래스 1..3 → 출력 채널 0..2).

**라벨링: CVAT, polyline.** 차선은 선이라 polyline으로 빠르게 라벨링하고, 이 노트북이
polyline을 **두께 `LANE_THICKNESS` px의 띠 마스크로 rasterize**한다 (semantic seg는
픽셀 영역이 필요하므로). polygon으로 면을 직접 칠하는 것과 학습 결과는 동일하되,
선 라벨이 빠르고 두께가 코드로 균일해진다.

사용 순서:
1. **런타임 → GPU (T4/L4)**
2. CVAT에서 차선 **polyline** 라벨링(left-solid/right-solid/center-dashed) →
   **Export → `CVAT for images 1.1`** → zip 다운로드 (`annotations.xml` + 원본 이미지)
3. 아래 **업로드 셀**에서 zip 업로드
4. `CONFIG`의 `CVAT_LABEL2ID`/`LANE_THICKNESS` 확인 후 실행
5. 끝나면 체크포인트 폴더(`segformer_lane/`)를 zip으로 받아 압축해제 후
   `extract_labels.py --segformer_ckpt segformer_lane` 로 사용

> `CVAT for images 1.1` export는 `annotations.xml`(polyline 좌표) + 원본 이미지를 담는다.
> 마스크는 export에 없고, 아래 셀이 polyline에서 즉석 생성한다.

## 1. 설치

In [ ]:
!pip install -q -U transformers datasets evaluate
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG

`ID2LABEL` 순서 고정 — extract_labels.py contract (배경 0, 좌실선 1, 우실선 2, 중앙점선 3).
`CVAT_LABEL2ID`는 CVAT polyline 라벨 이름(annotations.xml의 `label=`)을 이 id로 매핑한다
(이름이 다르면 여기만 수정). `LANE_THICKNESS`는 마스크 띠 두께.

In [ ]:
BASE_MODEL = 'nvidia/segformer-b0-finetuned-ade-512-512'  # b0 = Jetson 가벼움
IMG_SIZE   = 512        # SegFormer 학습 입력 (추론은 224 lane에 upsample되므로 무관)
EPOCHS     = 80
BATCH      = 8
LR         = 6e-5
OUT_DIR    = '/content/segformer_lane'
DATA_ROOT  = '/content/cvat_seg'   # 업로드한 zip을 풀 위치
VAL_FRAC   = 0.2                   # train/val split (CVAT는 split 없이 한 덩어리)
LANE_THICKNESS = 8                 # polyline → 마스크 띠 두께(px, 원본 해상도 기준)

# extract_labels.SegFormerLaneSeg와 반드시 일치
ID2LABEL = {0: 'background', 1: 'left-solid', 2: 'right-solid', 3: 'center-dashed'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

# CVAT polyline label 이름 → 우리 id. CVAT에서 라벨 이름을 다르게 지었으면 여기만 수정.
CVAT_LABEL2ID = {
    'left-solid':    1,
    'right-solid':   2,
    'center-dashed': 3,
}
print('id2label:', ID2LABEL)

## 3. CVAT zip 업로드 + 압축해제

In [ ]:
import os, glob, zipfile, shutil
from google.colab import files

shutil.rmtree(DATA_ROOT, ignore_errors=True)
os.makedirs(DATA_ROOT, exist_ok=True)
up = files.upload()                      # CVAT 'CVAT for images 1.1' zip 선택
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATA_ROOT)

# annotations.xml 위치 찾기 (CVAT export는 보통 루트 또는 한 단계 하위)
xmls = glob.glob(os.path.join(DATA_ROOT, '**', 'annotations.xml'), recursive=True)
assert xmls, 'annotations.xml 없음 — CVAT "CVAT for images 1.1" export인지 확인'
ANN_XML  = xmls[0]
SEG_ROOT = os.path.dirname(ANN_XML)
# 원본 이미지 폴더: CVAT은 images/ 아래 두는 경우가 많음 (없으면 SEG_ROOT 직속)
IMG_DIR = os.path.join(SEG_ROOT, 'images')
if not os.path.isdir(IMG_DIR):
    IMG_DIR = SEG_ROOT
print('annotations:', ANN_XML)
print('image dir  :', IMG_DIR)

## 4. annotations.xml 파싱 — polyline 좌표 추출

CVAT XML의 각 `<image>`마다 `<polyline label=... points="x,y;x,y;...">`를 읽어
(파일명 → [(class_id, [(x,y)...]) ...]) 매핑을 만든다.

In [ ]:
import xml.etree.ElementTree as ET

def parse_points(pstr):
    """'x1,y1;x2,y2;...' → [(x1,y1), (x2,y2), ...] (float)."""
    pts = []
    for tok in pstr.strip().split(';'):
        if not tok:
            continue
        x, y = tok.split(',')
        pts.append((float(x), float(y)))
    return pts

# image 파일명(basename) → list of (class_id, points)
ann = {}
tree = ET.parse(ANN_XML)
root = tree.getroot()
unknown_labels = set()
for img in root.findall('image'):
    name = os.path.basename(img.get('name'))
    lines = []
    for pl in img.findall('polyline'):
        label = pl.get('label')
        cid = CVAT_LABEL2ID.get(label)
        if cid is None:
            unknown_labels.add(label)
            continue
        lines.append((cid, parse_points(pl.get('points'))))
    ann[name] = lines

print('annotated images:', len(ann))
n_lines = sum(len(v) for v in ann.values())
print('total polylines :', n_lines)
if unknown_labels:
    print('!! CVAT_LABEL2ID에 없는 라벨(무시됨):', unknown_labels)
assert n_lines > 0, 'polyline이 하나도 안 잡힘 — 라벨 이름/ CVAT_LABEL2ID 확인'

In [ ]:
import glob, random
import cv2
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor

def polylines_to_mask(lines, h, w):
    """[(class_id, [(x,y)...]) ...] → (H,W) 클래스 인덱스맵 (0..3).
    각 polyline을 LANE_THICKNESS 두께로 그린다. 겹치면 나중 클래스가 덮음."""
    mask = np.zeros((h, w), dtype=np.uint8)
    for cid, pts in lines:
        if len(pts) < 2:
            continue
        arr = np.round(np.array(pts, dtype=np.float32)).astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(mask, [arr], isClosed=False, color=int(cid),
                      thickness=LANE_THICKNESS, lineType=cv2.LINE_8)
    return mask

# 이미지 파일과 annotation 매칭
def list_images():
    imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        imgs += glob.glob(os.path.join(IMG_DIR, '**', ext), recursive=True)
    return sorted(imgs)

pairs = [(p, ann[os.path.basename(p)]) for p in list_images()
         if os.path.basename(p) in ann]
assert pairs, 'annotations.xml의 이미지명과 실제 파일명이 안 맞음 — IMG_DIR 확인'
random.Random(0).shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_FRAC))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print('train pairs:', len(train_pairs), '| val pairs:', len(val_pairs))

processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)
processor.size = {'height': IMG_SIZE, 'width': IMG_SIZE}

class LaneSegDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_path, lines = self.pairs[i]
        image = Image.open(img_path).convert('RGB')
        w, h = image.size
        mask_idx = polylines_to_mask(lines, h, w)         # (H,W) 0..3, 원본 해상도
        enc = processor(image, Image.fromarray(mask_idx), return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

train_ds = LaneSegDataset(train_pairs)
val_ds   = LaneSegDataset(val_pairs)

## 5. 모델 + Trainer

In [ ]:
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer

model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,   # ADE20K head → 4클래스 head 교체
)

args = TrainingArguments(
    output_dir='/content/seg_ckpts',
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

In [ ]:
import evaluate
metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    # logits: (B, C, h, w) → 라벨 해상도로 upsample 후 argmax
    upsampled = torch.nn.functional.interpolate(
        logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
    preds = upsampled.argmax(dim=1).numpy()
    res = metric.compute(predictions=preds, references=labels,
                         num_labels=NUM_LABELS, ignore_index=255,
                         reduce_labels=False)
    return {'mean_iou': res['mean_iou'], 'mean_acc': res['mean_accuracy']}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

## 6. 체크포인트 저장 (extract_labels.py가 from_pretrained로 로드)

`model.save_pretrained` + `processor.save_pretrained`를 같은 폴더에 저장해야
`SegFormerLaneSeg(checkpoint_path)`가 그대로 로드한다.

In [ ]:
import shutil
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print('saved checkpoint ->', OUT_DIR)
print(os.listdir(OUT_DIR))   # config.json, model.safetensors, preprocessor_config.json 등

# zip으로 묶어 다운로드 (Drive 안 씀)
shutil.make_archive('/content/segformer_lane', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/segformer_lane.zip')

## 7. (선택) sanity check — 한 장 추론 시각화

In [ ]:
import matplotlib.pyplot as plt
model.eval()
img_path, lines = val_pairs[0]
image = Image.open(img_path).convert('RGB')
w, h = image.size
gt = polylines_to_mask(lines, h, w)            # rasterize 결과(GT) 눈 확인용

inp = processor(image, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**inp).logits
up = torch.nn.functional.interpolate(logits, size=image.size[::-1],
                                     mode='bilinear', align_corners=False)
pred = up.argmax(1)[0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(image); ax[0].set_title('lane'); ax[0].axis('off')
ax[1].imshow(gt,   vmin=0, vmax=NUM_LABELS-1, cmap='tab10')
ax[1].set_title('GT (polyline→mask)'); ax[1].axis('off')
ax[2].imshow(pred, vmin=0, vmax=NUM_LABELS-1, cmap='tab10')
ax[2].set_title('pred (0 bg / 1 L / 2 R / 3 dash)'); ax[2].axis('off')
plt.show()